
# GPT-5.4 Mini SFT Trajectory Generation

This notebook generates SlideVQA SFT trajectories with the OpenAI API and keeps only records that satisfy all filters:

1. all ground-truth `evidence_pages` were retrieved,
2. retrieval steps are at most 5,
3. GPT-5-mini judges the final answer equivalent to the reference answer.

No VISOR-style data augmentation is performed. Ground-truth pages and reference answers are used only for filtering and validation, not for query planning or page analysis.


In [ ]:

from pathlib import Path
import base64
import io
import json
import math
import os
import re
import statistics
import sys
import time
import urllib.error
import urllib.request
from collections import Counter
from typing import Any, Iterable

import numpy as np
from PIL import Image

PROJECT = Path('/root/autodl-tmp/visual_rag_agent')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))


def load_dotenv(path: Path = PROJECT / '.env', *, override: bool = False) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if override or key not in os.environ:
            os.environ[key] = value


load_dotenv()
DATASET_FILE = PROJECT / 'data/corpora/slidevqa/test.jsonl'
INDEX_DIR = PROJECT / 'data/indexes/slidevqa'
GENERATOR_MODEL = os.environ.get('OPENAI_SFT_GENERATOR_MODEL', 'gpt-5.4-mini')
VALIDATOR_MODEL = os.environ.get('OPENAI_SFT_VALIDATOR_MODEL', 'gpt-5-mini')
OPENAI_BASE_URL = os.environ.get('OPENAI_BASE_URL', 'https://api.openai.com/v1')
RETRIEVER_MODEL = os.environ.get('SFT_RETRIEVER_MODEL', '/root/autodl-tmp/models/Qwen3-VL-Embedding-8B')
RETRIEVER_DEVICE = os.environ.get('SFT_RETRIEVER_DEVICE', 'cuda')
RETRIEVER_DTYPE = os.environ.get('SFT_RETRIEVER_DTYPE', 'bfloat16')
RETRIEVER_ATTN = os.environ.get('SFT_RETRIEVER_ATTN', 'flash_attention_2')
START_INDEX = int(os.environ.get('SFT_START_INDEX', '0'))
MAX_SAMPLES = int(os.environ.get('SFT_MAX_SAMPLES', '20'))
MAX_RETRIEVAL_STEPS = 5
RETRIEVAL_TOP_K = int(os.environ.get('SFT_RETRIEVAL_TOP_K', '3'))
MAX_CONTEXT_IMAGES = int(os.environ.get('SFT_MAX_CONTEXT_IMAGES', '15'))
MAX_IMAGE_PIXELS = int(os.environ.get('SFT_IMAGE_MAX_PIXELS', '1250000'))
IMAGE_JPEG_QUALITY = int(os.environ.get('SFT_IMAGE_JPEG_QUALITY', '85'))
IMAGE_DETAIL = os.environ.get('SFT_IMAGE_DETAIL', 'high')
GENERATOR_TEMPERATURE = float(os.environ.get('SFT_GENERATOR_TEMPERATURE', '0.2'))
REQUEST_TIMEOUT = float(os.environ.get('OPENAI_REQUEST_TIMEOUT', '120'))
MAX_RETRIES = int(os.environ.get('OPENAI_MAX_RETRIES', '3'))
RUN_GENERATION = os.environ.get('RUN_OPENAI_SFT_GENERATION', '0') == '1'
RUN_ID = os.environ.get('SFT_RUN_ID', time.strftime('%Y%m%d_%H%M%S'))
OUTPUT_DIR = Path(os.environ.get('SFT_TRAJECTORY_OUTPUT_DIR', str(PROJECT / 'outputs/sft_trajectories' / f'gpt54mini_{RUN_ID}')))
RAW_PATH = OUTPUT_DIR / 'raw_trajectories.jsonl'
KEPT_PATH = OUTPUT_DIR / 'kept_sft_trajectories.jsonl'
REJECTED_PATH = OUTPUT_DIR / 'rejected_trajectories.jsonl'
SUMMARY_PATH = OUTPUT_DIR / 'summary.json'

print('OPENAI_API_KEY:', 'set' if os.environ.get('OPENAI_API_KEY') else 'missing')
print('generator:', GENERATOR_MODEL)
print('validator:', VALIDATOR_MODEL)
print('retriever:', RETRIEVER_MODEL)
print('dataset/index:', DATASET_FILE.exists(), INDEX_DIR.exists())
print('run generation:', RUN_GENERATION)
print('output dir:', OUTPUT_DIR)


def read_jsonl(path: Path, *, limit: int | None = None, start: int = 0) -> list[dict[str, Any]]:
    rows = []
    seen = 0
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if seen < start:
                seen += 1
                continue
            rows.append(json.loads(line))
            seen += 1
            if limit is not None and len(rows) >= limit:
                break
    return rows


def write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')


def first_present(row: dict[str, Any], names: Iterable[str], default: Any = None) -> Any:
    for name in names:
        value = row.get(name)
        if value not in (None, ''):
            return value
    return default


def extract_question(row: dict[str, Any]) -> str:
    return str(first_present(row, ('query', 'question', 'prompt', 'input', 'problem'), '')).strip()


def extract_answer(row: dict[str, Any]) -> str:
    value = first_present(row, ('answer', 'answers', 'target', 'label', 'response'), '')
    return ' | '.join(str(item) for item in value) if isinstance(value, list) else str(value).strip()


def extract_sample_id(row: dict[str, Any], row_index: int) -> str:
    return str(first_present(row, ('id', 'qid', 'question_id', 'uid', 'eval_id', 'qa_id'), row_index))


def extract_deck_name(row: dict[str, Any]) -> str:
    return str(first_present(row, ('deck_name', 'deck_id', 'doc_id', 'document_id', 'pdf_id'), '')).strip()


def page_num_to_label(deck_name: str, page_num: int) -> str:
    return f'{deck_name}/page_{page_num:02d}' if deck_name else f'page_{page_num:02d}'


def label_to_page_num(label: str) -> int | None:
    match = re.search(r'page[_-]?(\d+)', str(label), flags=re.I)
    return int(match.group(1)) if match else None


def extract_reference_pages(row: dict[str, Any]) -> set[int]:
    value = first_present(row, ('reference_pages', 'gold_pages', 'evidence_pages', 'page_ids', 'pages', 'answer_pages'), [])
    if isinstance(value, str):
        return {int(num) for num in re.findall(r'\d+', value)}
    if isinstance(value, (int, float)):
        return {int(value)}
    out = set()
    if isinstance(value, list):
        for item in value:
            if isinstance(item, dict):
                item = first_present(item, ('page', 'page_num', 'page_id', 'index'), None)
            try:
                out.add(int(item))
            except (TypeError, ValueError):
                pass
    return out


def extract_reference_labels(row: dict[str, Any]) -> set[str]:
    deck_name = extract_deck_name(row)
    return {page_num_to_label(deck_name, page_num) for page_num in extract_reference_pages(row)}

for idx, row in enumerate(read_jsonl(DATASET_FILE, limit=3)):
    print(idx, extract_sample_id(row, idx), sorted(extract_reference_labels(row)), extract_answer(row))


In [ ]:

class OpenAIAPIError(RuntimeError):
    def __init__(self, status: int | None, body: str) -> None:
        super().__init__(f'OpenAI API error status={status}: {body[:1200]}')
        self.status = status
        self.body = body


class OpenAIResponsesHTTPClient:
    def __init__(self, *, api_key: str, base_url: str, timeout: float, max_retries: int) -> None:
        self.api_key = api_key
        self.base_url = base_url.rstrip('/')
        self.timeout = timeout
        self.max_retries = max_retries

    def create_response(self, *, model: str, input_items: list[dict[str, Any]], max_output_tokens: int, temperature: float | None, text_format: dict[str, Any] | None = None) -> dict[str, Any]:
        payload = {'model': model, 'input': input_items, 'max_output_tokens': max_output_tokens}
        if temperature is not None:
            payload['temperature'] = temperature
        if text_format is not None:
            payload['text'] = {'format': text_format}
        try:
            return self._post('/responses', payload)
        except OpenAIAPIError as exc:
            if exc.status == 400 and 'temperature' in exc.body.lower():
                payload.pop('temperature', None)
                return self._post('/responses', payload)
            raise

    def _post(self, path: str, payload: dict[str, Any]) -> dict[str, Any]:
        data = json.dumps(payload).encode('utf-8')
        last_error = None
        for attempt in range(self.max_retries + 1):
            req = urllib.request.Request(f'{self.base_url}{path}', data=data, headers={'Authorization': f'Bearer {self.api_key}', 'Content-Type': 'application/json'}, method='POST')
            try:
                with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                    return json.loads(resp.read().decode('utf-8'))
            except urllib.error.HTTPError as exc:
                body = exc.read().decode('utf-8', errors='replace')
                last_error = OpenAIAPIError(exc.code, body)
                if exc.code not in {408, 409, 429, 500, 502, 503, 504}:
                    raise last_error
            except Exception as exc:
                last_error = exc
            if attempt < self.max_retries:
                time.sleep(min(20, 1.5 * (attempt + 1)))
        if isinstance(last_error, OpenAIAPIError):
            raise last_error
        raise OpenAIAPIError(None, str(last_error))


def extract_response_text(resp: dict[str, Any]) -> str:
    if isinstance(resp.get('output_text'), str):
        return resp['output_text'].strip()
    pieces = []
    for item in resp.get('output', []) or []:
        for content in item.get('content', []) or []:
            text = content.get('text') if isinstance(content, dict) else None
            if isinstance(text, str):
                pieces.append(text)
    return '\n'.join(pieces).strip()


def extract_json_object(text: str) -> dict[str, Any]:
    stripped = text.strip()
    if stripped.startswith('```'):
        stripped = stripped.strip('`')
        if stripped.lower().startswith('json'):
            stripped = stripped[4:].strip()
    try:
        return json.loads(stripped)
    except json.JSONDecodeError:
        pass
    start, end = stripped.find('{'), stripped.rfind('}')
    if start < 0 or end < start:
        raise ValueError(f'No JSON object found: {text[:400]}')
    return json.loads(stripped[start:end + 1])


def image_to_data_url(path: str | Path) -> str:
    with Image.open(path) as image:
        image = image.convert('RGB')
        w, h = image.size
        if MAX_IMAGE_PIXELS > 0 and w * h > MAX_IMAGE_PIXELS:
            scale = math.sqrt(MAX_IMAGE_PIXELS / float(w * h))
            image = image.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.Resampling.LANCZOS)
        buf = io.BytesIO()
        image.save(buf, format='JPEG', quality=IMAGE_JPEG_QUALITY, optimize=True)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')


def message(role: str, text: str, image_paths: list[str] | None = None) -> dict[str, Any]:
    content = [{'type': 'input_text', 'text': text}]
    for image_path in image_paths or []:
        content.append({'type': 'input_image', 'image_url': image_to_data_url(image_path), 'detail': IMAGE_DETAIL})
    return {'role': role, 'content': content}


def call_json(client: OpenAIResponsesHTTPClient, *, model: str, system: str, user: str, schema_spec: dict[str, Any], image_paths: list[str] | None = None, max_output_tokens: int = 700, temperature: float | None = 0.0) -> tuple[dict[str, Any], str, dict[str, Any]]:
    fmt = {'type': 'json_schema', 'name': schema_spec['name'], 'schema': schema_spec['schema'], 'strict': True}
    try:
        resp = client.create_response(model=model, input_items=[message('system', system), message('user', user, image_paths)], max_output_tokens=max_output_tokens, temperature=temperature, text_format=fmt)
    except OpenAIAPIError as exc:
        body = exc.body.lower()
        if 'json_schema' not in body and 'text.format' not in body and 'structured' not in body:
            raise
        user = user + '\n\nReturn JSON only matching this schema:\n' + json.dumps(schema_spec['schema'], ensure_ascii=False)
        resp = client.create_response(model=model, input_items=[message('system', system), message('user', user, image_paths)], max_output_tokens=max_output_tokens, temperature=temperature, text_format=None)
    raw = extract_response_text(resp)
    return extract_json_object(raw), raw, resp


def compact_usage(resp: dict[str, Any]) -> dict[str, Any]:
    usage = resp.get('usage') or {}
    return usage if isinstance(usage, dict) else {}


In [ ]:

from src.retriever import Retriever, normalize_rows


class DeckRestrictedSearcher:
    def __init__(self) -> None:
        self.retriever = Retriever(model_path=RETRIEVER_MODEL, index_path=INDEX_DIR, device=RETRIEVER_DEVICE, dtype=RETRIEVER_DTYPE, attn_implementation=RETRIEVER_ATTN, load_model=True)
        if self.retriever.embeddings is None or not self.retriever.entries:
            raise RuntimeError(f'Missing retriever index under {INDEX_DIR}')

    def search(self, *, query: str, deck_name: str, top_k: int, banned: set[str]) -> list[dict[str, Any]]:
        if self.retriever.embedder is None:
            raise RuntimeError('retriever embedder not loaded')
        q = self.retriever.embedder.embed_texts([query])[0]
        q = normalize_rows(q[None, :])[0]
        scores = self.retriever.embeddings @ q
        ranked = np.argsort(-scores)
        out = []
        prefix = f'{deck_name}/' if deck_name else ''
        for raw_idx in ranked:
            entry = self.retriever.entries[int(raw_idx)]
            label = entry.page_label
            if label in banned:
                continue
            if prefix and not label.startswith(prefix):
                continue
            image_path = Path(entry.image_path)
            if not image_path.is_absolute():
                image_path = PROJECT / image_path
            if not image_path.exists():
                continue
            out.append({'page_label': label, 'page_num': label_to_page_num(label), 'image_path': str(image_path), 'score': float(scores[int(raw_idx)])})
            if len(out) >= top_k:
                break
        return out


In [ ]:

PLANNER_SYSTEM = 'You are a visual-RAG query planner for SlideVQA. Use only question, retrieved-page memory, and search trace. Return JSON only.'
ANALYSIS_SYSTEM = 'You are a visual SlideVQA page analyst. For each retrieved slide image, summarize visible evidence and extract question-relevant facts. Do not use hidden labels, ground-truth pages, or reference answers. Return JSON only.'
ANSWER_SYSTEM = 'Answer a SlideVQA question from retrieved slide images and memory. Use only provided images and memory. Cite supporting page labels. Return JSON only.'
VALIDATOR_SYSTEM = 'You are a strict but reasonable SlideVQA answer-equivalence judge. Accept equivalent numeric/unit formatting. Reject wrong values, wrong units, missing answers, and unsupported related discussion. Return JSON only.'

PLANNER_SCHEMA = {'name': 'sft_trajectory_planner', 'schema': {'type': 'object', 'additionalProperties': False, 'properties': {'action': {'type': 'string', 'enum': ['search', 'answer']}, 'query': {'type': 'string'}, 'rationale': {'type': 'string'}, 'answer_draft': {'type': 'string'}}, 'required': ['action', 'query', 'rationale', 'answer_draft']}}
ANALYSIS_SCHEMA = {'name': 'sft_page_analysis', 'schema': {'type': 'object', 'additionalProperties': False, 'properties': {'page_analyses': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': False, 'properties': {'page_label': {'type': 'string'}, 'summary': {'type': 'string'}, 'contains_answer_evidence': {'type': 'boolean'}, 'key_facts': {'type': 'array', 'items': {'type': 'string'}}, 'answer_candidate': {'type': 'string'}, 'missing_info': {'type': 'string'}, 'next_query_hint': {'type': 'string'}}, 'required': ['page_label', 'summary', 'contains_answer_evidence', 'key_facts', 'answer_candidate', 'missing_info', 'next_query_hint']}}, 'memory_update': {'type': 'string'}, 'ready_to_answer': {'type': 'boolean'}}, 'required': ['page_analyses', 'memory_update', 'ready_to_answer']}}
ANSWER_SCHEMA = {'name': 'sft_final_answer', 'schema': {'type': 'object', 'additionalProperties': False, 'properties': {'final_answer': {'type': 'string'}, 'cited_page_labels': {'type': 'array', 'items': {'type': 'string'}}, 'rationale': {'type': 'string'}}, 'required': ['final_answer', 'cited_page_labels', 'rationale']}}
JUDGE_SCHEMA = {'name': 'sft_answer_equivalence_judge', 'schema': {'type': 'object', 'additionalProperties': False, 'properties': {'correct': {'type': 'boolean'}, 'score': {'type': 'integer', 'enum': [0, 1]}, 'rationale': {'type': 'string'}, 'normalized_prediction': {'type': 'string'}, 'normalized_reference': {'type': 'string'}}, 'required': ['correct', 'score', 'rationale', 'normalized_prediction', 'normalized_reference']}}


def render_memory(obs: list[dict[str, Any]], labels: list[str], max_chars: int = 5000) -> str:
    if not obs:
        return 'No retrieved page memory yet.'
    lines = []
    for item in obs[-12:]:
        facts = '; '.join(item.get('key_facts') or [])
        lines.append(f"- {item.get('page_label')}: evidence={item.get('contains_answer_evidence')}; summary={item.get('summary', '')}; facts={facts}; answer_candidate={item.get('answer_candidate', '')}; missing={item.get('missing_info', '')}")
    text = '\n'.join(lines)
    return f"Retrieved pages so far: {', '.join(labels[-20:]) or 'None'}\nPage memory:\n{text[-max_chars:]}"


def plan_next(client, question, obs, labels, queries, step):
    user = '\n'.join(['Question:', question, '', 'Current memory:', render_memory(obs, labels), '', 'Issued queries:', json.dumps(queries or ['None'], ensure_ascii=False), '', f'Step {step} of at most {MAX_RETRIEVAL_STEPS}. If evidence is sufficient, action=answer with answer_draft. Otherwise action=search with one concise retrieval query.'])
    return call_json(client, model=GENERATOR_MODEL, system=PLANNER_SYSTEM, user=user, schema_spec=PLANNER_SCHEMA, max_output_tokens=350, temperature=GENERATOR_TEMPERATURE)


def analyse_pages(client, question, query, pages, obs, labels):
    label_block = '\n'.join(f'[{i}] {p["page_label"]}' for i, p in enumerate(pages, 1))
    user = '\n'.join(['Question:', question, '', 'Retrieval query:', query, '', 'Current memory:', render_memory(obs, labels), '', 'Images are in this order:', label_block, '', 'Return one analysis object for every page_label exactly as listed.'])
    return call_json(client, model=GENERATOR_MODEL, system=ANALYSIS_SYSTEM, user=user, schema_spec=ANALYSIS_SCHEMA, image_paths=[p['image_path'] for p in pages], max_output_tokens=1200, temperature=GENERATOR_TEMPERATURE)


def normalize_analyses(payload, pages):
    by_label = {str(x.get('page_label', '')).strip(): x for x in payload.get('page_analyses', []) if isinstance(x, dict)}
    out = []
    for page in pages:
        label = page['page_label']
        item = dict(by_label.get(label) or {})
        for k, v in {'page_label': label, 'summary': '', 'contains_answer_evidence': False, 'key_facts': [], 'answer_candidate': '', 'missing_info': '', 'next_query_hint': ''}.items():
            item.setdefault(k, v)
        item.update({'image_path': page['image_path'], 'page_num': page.get('page_num'), 'retrieval_score': page.get('score')})
        out.append(item)
    return out


def choose_answer_pages(obs, pages_by_label, labels):
    selected = []
    for item in obs:
        if item.get('contains_answer_evidence') or item.get('key_facts') or item.get('answer_candidate'):
            label = item.get('page_label')
            if label and label not in selected:
                selected.append(label)
    for label in labels:
        if label not in selected:
            selected.append(label)
    return [pages_by_label[x] for x in selected[:MAX_CONTEXT_IMAGES] if x in pages_by_label]


def final_answer(client, question, obs, pages_by_label, labels):
    pages = choose_answer_pages(obs, pages_by_label, labels)
    label_block = '\n'.join(f'[{i}] {p["page_label"]}' for i, p in enumerate(pages, 1))
    user = '\n'.join(['Question:', question, '', 'Memory:', render_memory(obs, labels, 7000), '', 'Attached answer-context images:', label_block or 'None'])
    payload, raw, resp = call_json(client, model=GENERATOR_MODEL, system=ANSWER_SYSTEM, user=user, schema_spec=ANSWER_SCHEMA, image_paths=[p['image_path'] for p in pages], max_output_tokens=500, temperature=0.0)
    return payload, raw, resp, pages


def judge_answer(client, question, reference_answer, prediction, sample_id):
    user = json.dumps({'sample_id': sample_id, 'question': question, 'reference_answer': reference_answer, 'prediction': prediction}, ensure_ascii=False, indent=2)
    return call_json(client, model=VALIDATOR_MODEL, system=VALIDATOR_SYSTEM, user=user, schema_spec=JUDGE_SCHEMA, max_output_tokens=450, temperature=0.0)


def build_sft_messages(record):
    messages = [{'role': 'system', 'content': 'You are a visual-RAG agent. Search pages, inspect retrieved images, keep concise evidence memory, and answer only when evidence is sufficient.'}, {'role': 'user', 'content': record['question']}]
    for step in record.get('trace', []):
        if step.get('type') == 'planner':
            messages.append({'role': 'assistant', 'content': json.dumps(step.get('planner', {}), ensure_ascii=False)})
        elif step.get('type') == 'search':
            messages.append({'role': 'tool', 'name': 'retrieve_pages', 'content': json.dumps({'query': step.get('query'), 'pages': step.get('pages', [])}, ensure_ascii=False)})
        elif step.get('type') == 'analysis':
            messages.append({'role': 'assistant', 'content': json.dumps({'page_analyses': step.get('page_analyses', []), 'memory_update': step.get('memory_update', ''), 'ready_to_answer': step.get('ready_to_answer', False)}, ensure_ascii=False)})
    messages.append({'role': 'assistant', 'content': record.get('final_answer', '')})
    return messages


In [ ]:

def run_one_example(client, searcher, row, row_index):
    sample_id = extract_sample_id(row, row_index)
    question = extract_question(row)
    reference_answer = extract_answer(row)
    deck = extract_deck_name(row)
    reference_labels = extract_reference_labels(row)
    trace, usage, obs, labels, queries = [], [], [], [], []
    pages_by_label = {}
    stop_reason = 'max_retrieval_steps'
    for step in range(1, MAX_RETRIEVAL_STEPS + 1):
        planner, planner_raw, planner_resp = plan_next(client, question, obs, labels, queries, step)
        usage.append({'step': step, 'call': 'planner', 'usage': compact_usage(planner_resp)})
        trace.append({'type': 'planner', 'step': step, 'planner': planner, 'raw_text': planner_raw, 'response_id': planner_resp.get('id')})
        if str(planner.get('action', 'search')).lower() == 'answer' and obs:
            stop_reason = 'planner_answer'
            break
        query = str(planner.get('query') or '').strip() or question
        queries.append(query)
        pages = searcher.search(query=query, deck_name=deck, top_k=RETRIEVAL_TOP_K, banned=set(labels))
        labels.extend(p['page_label'] for p in pages)
        for page in pages:
            pages_by_label[page['page_label']] = page
        trace.append({'type': 'search', 'step': step, 'query': query, 'pages': pages})
        if not pages:
            continue
        analysis, analysis_raw, analysis_resp = analyse_pages(client, question, query, pages, obs, labels)
        usage.append({'step': step, 'call': 'analysis', 'usage': compact_usage(analysis_resp)})
        page_analyses = normalize_analyses(analysis, pages)
        obs.extend(page_analyses)
        trace.append({'type': 'analysis', 'step': step, 'query': query, 'page_analyses': page_analyses, 'memory_update': analysis.get('memory_update', ''), 'ready_to_answer': bool(analysis.get('ready_to_answer')), 'raw_text': analysis_raw, 'response_id': analysis_resp.get('id')})
        if analysis.get('ready_to_answer'):
            stop_reason = 'analysis_ready_to_answer'
            break
    answer_payload, answer_raw, answer_resp, answer_pages = final_answer(client, question, obs, pages_by_label, labels)
    final = str(answer_payload.get('final_answer', '')).strip()
    usage.append({'step': None, 'call': 'final_answer', 'usage': compact_usage(answer_resp)})
    judge, judge_raw, judge_resp = judge_answer(client, question, reference_answer, final, sample_id)
    usage.append({'step': None, 'call': 'validator', 'usage': compact_usage(judge_resp)})
    retrieved = set(labels)
    missing = sorted(reference_labels - retrieved)
    retrieval_steps = sum(1 for x in trace if x.get('type') == 'search')
    checks = {'all_reference_pages_retrieved': bool(reference_labels) and not missing, 'retrieval_steps_ok': retrieval_steps <= MAX_RETRIEVAL_STEPS, 'answer_matches_reference': bool(judge.get('correct')) and int(judge.get('score', 0)) == 1}
    reasons = []
    if not reference_labels: reasons.append('no_reference_pages')
    if not checks['all_reference_pages_retrieved']: reasons.append('missing_reference_pages')
    if not checks['retrieval_steps_ok']: reasons.append('too_many_retrieval_steps')
    if not checks['answer_matches_reference']: reasons.append('answer_mismatch')
    record = {'sample_id': sample_id, 'row_index': row_index, 'deck_name': deck, 'question': question, 'reference_answer': reference_answer, 'reference_page_labels': sorted(reference_labels), 'retrieved_page_labels': labels, 'found_reference_page_labels': sorted(reference_labels & retrieved), 'missing_reference_page_labels': missing, 'retrieval_steps': retrieval_steps, 'stop_reason': stop_reason, 'final_answer': final, 'final_answer_payload': answer_payload, 'validator': {'provider': 'openai', 'model': VALIDATOR_MODEL, **judge}, 'keep_checks': checks, 'keep': all(checks.values()), 'reject_reasons': reasons, 'answer_context_pages': answer_pages, 'observations': obs, 'trace': trace, 'api': {'generator_model': GENERATOR_MODEL, 'validator_model': VALIDATOR_MODEL, 'retriever_model': RETRIEVER_MODEL, 'base_url': OPENAI_BASE_URL, 'usage': usage, 'answer_raw_text': answer_raw, 'validator_raw_text': judge_raw, 'answer_response_id': answer_resp.get('id'), 'validator_response_id': judge_resp.get('id')}}
    record['sft_messages'] = build_sft_messages(record)
    return record


def summarize_records(records):
    generated = [x for x in records if not x.get('error')]
    kept = [x for x in generated if x.get('keep')]
    rejected = [x for x in generated if not x.get('keep')]
    reasons = Counter(r for x in rejected for r in x.get('reject_reasons', []))
    return {'num_records': len(records), 'num_generated': len(generated), 'num_errors': sum(1 for x in records if x.get('error')), 'num_kept': len(kept), 'keep_rate': len(kept) / len(generated) if generated else 0, 'reject_reason_counts': dict(reasons), 'mean_retrieval_steps_generated': statistics.mean([x.get('retrieval_steps', 0) for x in generated]) if generated else 0, 'generator_model': GENERATOR_MODEL, 'validator_model': VALIDATOR_MODEL, 'retriever_model': RETRIEVER_MODEL, 'max_retrieval_steps': MAX_RETRIEVAL_STEPS, 'retrieval_top_k': RETRIEVAL_TOP_K, 'validation_mechanism': 'OpenAI GPT-5-mini JSON judge over question, reference_answer, and final_answer'}


if not RUN_GENERATION:
    print('Generation disabled. Set RUN_OPENAI_SFT_GENERATION=1 after setting OPENAI_API_KEY.')
elif not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY missing')
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    rows = read_jsonl(DATASET_FILE, limit=MAX_SAMPLES, start=START_INDEX)
    existing = read_jsonl(RAW_PATH) if RAW_PATH.exists() else []
    done = {str(x.get('sample_id')) for x in existing}
    searcher = DeckRestrictedSearcher()
    client = OpenAIResponsesHTTPClient(api_key=os.environ['OPENAI_API_KEY'], base_url=OPENAI_BASE_URL, timeout=REQUEST_TIMEOUT, max_retries=MAX_RETRIES)
    for local_idx, row in enumerate(rows):
        row_index = START_INDEX + local_idx
        sample_id = extract_sample_id(row, row_index)
        if sample_id in done:
            continue
        started = time.time()
        try:
            record = run_one_example(client, searcher, row, row_index)
        except Exception as exc:
            record = {'sample_id': sample_id, 'row_index': row_index, 'question': extract_question(row), 'reference_answer': extract_answer(row), 'reference_page_labels': sorted(extract_reference_labels(row)), 'keep': False, 'reject_reasons': ['error'], 'error': f'{type(exc).__name__}: {exc}'}
        record['elapsed_sec'] = round(time.time() - started, 2)
        append_jsonl(RAW_PATH, record)
        done.add(sample_id)
        print(json.dumps({'done': len(done), 'sample_id': sample_id, 'keep': record.get('keep'), 'reject_reasons': record.get('reject_reasons'), 'retrieval_steps': record.get('retrieval_steps'), 'elapsed_sec': record.get('elapsed_sec'), 'error': record.get('error')}, ensure_ascii=False), flush=True)
    all_records = read_jsonl(RAW_PATH)
    write_jsonl(KEPT_PATH, [x for x in all_records if x.get('keep')])
    write_jsonl(REJECTED_PATH, [x for x in all_records if not x.get('keep')])
    summary = summarize_records(all_records)
    SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps({'output_dir': str(OUTPUT_DIR), 'summary': summary}, ensure_ascii=False, indent=2))


In [ ]:

def validate_kept_record_shape(record):
    required = ['sample_id', 'question', 'reference_answer', 'reference_page_labels', 'retrieved_page_labels', 'retrieval_steps', 'final_answer', 'validator', 'keep_checks', 'trace', 'sft_messages']
    for key in required:
        assert key in record, f'missing {key}'
    assert record['keep'] is True
    assert record['keep_checks']['all_reference_pages_retrieved'] is True
    assert record['keep_checks']['retrieval_steps_ok'] is True
    assert record['keep_checks']['answer_matches_reference'] is True
    assert set(record['reference_page_labels']).issubset(set(record['retrieved_page_labels']))
    assert int(record['retrieval_steps']) <= MAX_RETRIEVAL_STEPS
    assert record['validator']['model'] == VALIDATOR_MODEL
    assert isinstance(record['sft_messages'], list) and record['sft_messages']

if RAW_PATH.exists():
    records = read_jsonl(RAW_PATH)
    kept = [x for x in records if x.get('keep')]
    print(json.dumps(summarize_records(records), ensure_ascii=False, indent=2))
    for item in kept:
        validate_kept_record_shape(item)
    print('validated kept records:', len(kept))
    if kept:
        first = kept[0]
        preview = {'sample_id': first['sample_id'], 'question': first['question'], 'reference_answer': first['reference_answer'], 'final_answer': first['final_answer'], 'reference_page_labels': first['reference_page_labels'], 'retrieved_page_labels': first['retrieved_page_labels'], 'validator': first['validator'], 'sft_messages_preview': first['sft_messages'][:5]}
        print(json.dumps(preview, ensure_ascii=False, indent=2)[:8000])
else:
    print('No raw trajectory file yet:', RAW_PATH)
